In [3]:
# ── Imports ───────────────────────────────────────────────────────────────────
# Standard toolkit imports for data loading, feature building, backtesting,
# and MLflow logging. LightGBM is used as the model.

from portfolio_toolkit import (
    load_prices,
    build_features,
    make_forward_return_target,
    make_forward_realized_vol_target,
    slice_split,
    validate_prediction_frame,
    weights_from_predictions_risk_adjusted,
    backtest_weights,
    write_backtest_artifacts,
    init_mlflow,
    start_run,
    log_predictions,
    log_portfolio,
    log_backtest,
)
import lightgbm as lgb
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

In [4]:
# ── Repo Root ─────────────────────────────────────────────────────────────────
# Since this notebook lives in MODELS/Brian/, we need to point the toolkit
# back to the repo root so it can find configs and cached data.

from pathlib import Path
repo_root = Path("../../").resolve()
print("Repo root:", repo_root)

Repo root: C:\Users\brixn\Documents\Portfolio-Optimization-Lib


In [5]:
# ── Load Shared Dataset ────────────────────────────────────────────────────────
# Using shared_set_2 (growth/tech universe: AAPL, MSFT, NVDA, etc.)
# Prices are cached locally after first download.

dataset_name = "shared_set_2"
prices = load_prices(dataset_name, repo_root=repo_root)
print("Prices loaded:", prices.shape)

Prices loaded: (78468, 8)


In [6]:
# ── Feature Engineering ────────────────────────────────────────────────────────
# Using 6 built-in toolkit features covering momentum, volatility, trend,
# oscillators, volume, and benchmark-relative performance.
# Custom feature: mom_vol_ratio = risk-adjusted momentum (momentum / volatility)
# This captures how strong the momentum signal is relative to how noisy the stock is.

features = build_features(prices, feature_names=[
    "momentum_20d",
    "vol_20d",
    "price_to_sma_20d",
    "rsi_14",
    "volume_zscore_20d",
    "excess_return_20d_vs_spy",
])

# Custom notebook-local feature
features["mom_vol_ratio"] = (
    features["momentum_20d"] / features["vol_20d"].replace(0, float("nan"))
)

# Build return and volatility targets (5-day forward horizon)
target_return = make_forward_return_target(prices, horizon=5)
target_vol = make_forward_realized_vol_target(prices, window=5)

# Merge features and targets into one dataset
dataset = features.merge(target_return, on=["date", "ticker"], how="left")
dataset = dataset.merge(target_vol, on=["date", "ticker"], how="left")
dataset = dataset.dropna()
print("Dataset shape:", dataset.shape)

Dataset shape: (77805, 11)


In [8]:
# ── Train / Val / Test Split ───────────────────────────────────────────────────
# Using shared split boundaries from the toolkit config:
# Train: 2014-2019 | Val: 2020-2021 | Test: 2022-2025

train = slice_split(dataset, dataset_name, "train", repo_root=repo_root)
val   = slice_split(dataset, dataset_name, "val", repo_root=repo_root)
test  = slice_split(dataset, dataset_name, "test", repo_root=repo_root)
print("Train:", train.shape, "Val:", val.shape, "Test:", test.shape)

Train: (38735, 11) Val: (13130, 11) Test: (25940, 11)


In [9]:
# ── Train LightGBM Model ───────────────────────────────────────────────────────
# Baseline LightGBM regression model predicting 5-day forward returns.
# Early stopping on validation RMSE to prevent overfitting.

feature_cols = [
    "momentum_20d",
    "vol_20d",
    "price_to_sma_20d",
    "rsi_14",
    "volume_zscore_20d",
    "excess_return_20d_vs_spy",
    "mom_vol_ratio",
]

X_train = train[feature_cols].fillna(0.0)
y_train = train["forward_return_5d"]
X_val   = val[feature_cols].fillna(0.0)
y_val   = val["forward_return_5d"]
X_test  = test[feature_cols].fillna(0.0)

train_data = lgb.Dataset(X_train, label=y_train)
val_data   = lgb.Dataset(X_val, label=y_val, reference=train_data)

# Baseline params - to be tuned in future iterations
params = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "verbose": -1,
}

model = lgb.train(
    params,
    train_data,
    num_boost_round=500,
    valid_sets=[val_data],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
)

print("Best iteration:", model.best_iteration)

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[2]	valid_0's rmse: 0.0555497
Best iteration: 2


In [11]:
# ── Generate Predictions ───────────────────────────────────────────────────────
# Predict expected returns and volatility for all test set rows.
# Validated against the shared prediction contract before backtesting.

predictions = test[["date", "ticker"]].copy()
predictions["horizon"] = 5
predictions["expected_return"] = model.predict(X_test)
predictions["expected_volatility"] = test["forward_realized_vol_5d"]

predictions = validate_prediction_frame(predictions, dataset_name=dataset_name, horizon=5, repo_root=repo_root)
print("Predictions validated:", predictions.shape)

Predictions validated: (25940, 5)


In [15]:
# ── Portfolio Construction + Backtest ──────────────────────────────────────────
# Risk-adjusted portfolio builder uses both expected_return and expected_volatility
# to construct weights that optimize for Sharpe ratio.
# Artifacts written to runs/Brian_Init including QuantStats HTML report.

portfolio = weights_from_predictions_risk_adjusted(predictions, dataset_name=dataset_name)
result = backtest_weights(dataset_name, portfolio, repo_root=repo_root)
artifacts = write_backtest_artifacts(result, "runs/Brian_Init")
print("Sharpe:", result.metrics.get("sharpe"))
print("Annual Return:", result.metrics.get("annual_return"))
print("Artifacts:", list(artifacts.keys()))

Sharpe: 0.14898734137258957
Annual Return: 0.02075726910598341
Artifacts: ['weights', 'nav', 'returns', 'turnover', 'benchmarks', 'metrics', 'metrics_table', 'quantstats_report']


In [16]:
# ── Log to MLflow ──────────────────────────────────────────────────────────────
# Logs predictions, portfolio weights, and backtest metrics to the shared
# MLflow server. Run name: Brian_Init (baseline LightGBM run).

init_mlflow(repo_root=repo_root)

with start_run(
    run_name="Brian_Init",
    dataset_name=dataset_name,
    tags={"model_type": "lightgbm", "horizon": "5", "version": "baseline"},
    repo_root=repo_root,
):
    log_predictions(predictions)
    log_portfolio(portfolio)
    log_backtest(result)

print("Logged to MLflow successfully!")

🏃 View run Brian_Init at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/1/runs/c7a2e31afd9c4791a7c31a8c77206ce0
🧪 View experiment at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/1
Logged to MLflow successfully!
